In [1]:
# imports
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBClassifier
import shap

c:\Users\will6\miniconda3\envs\cs320\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#### BASE

#read csv
training = pd.read_csv("train_data_m.csv")

#features
features = ['seed_dif', 
            'massey_rank_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            #'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            #'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]


#Initialize arrays
training_dfs = []
base_val_dfs = []
val_years = []
years = [2011,2012,2013,2014,2015,2016,2017,2018,2019,2021,2022,2023,2024]

#Validation by year
#We train only on data before validation year
for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    temp_training = training.query("Season <= @i")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    base_val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

#array for loss
base_val_error = []

#model
base_mod = XGBClassifier(n_estimators=600, max_depth=3, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 12, max_bin = 20, min_child_weight = 2, num_parallel_tree = 1, objective='binary:logistic', seed = 323)

for i in range(len(years)):
    X_train = training_dfs[i][features]
    y_train = training_dfs[i]['result']

    val = base_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    base_mod.fit(X_train, y_train)
    
    #make predictions
    preds = base_mod.predict_proba(X_val)
    val['pred'] = preds[:,1]
    val['loss'] = (val['pred'] - val['result'])**2

    base_val_dfs[i] = val.copy()
    print(val_years[i], "Loss:", np.mean(val['loss']))
    base_val_error.append(np.mean(val['loss']))

print(np.mean(base_val_error[-5:]))

#print(val_base.drop('loss', axis = 1).sort_values('pred', ascending = False).head(5))
#val_base.drop('loss', axis = 1).sort_values('pred', ascending = True).head(5)

#shap
explainer = shap.TreeExplainer(base_mod)
shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)


2012 Loss: 0.1974814012860625
2013 Loss: 0.20004218705746035
2014 Loss: 0.20425875563683518
2015 Loss: 0.17312351795592454
2016 Loss: 0.19560346521596406
2017 Loss: 0.1835725788662484
2018 Loss: 0.202243712122715
2019 Loss: 0.17347296459017142
2021 Loss: 0.2211849136860186
2022 Loss: 0.22263964472302863
2023 Loss: 0.19897605645861757
2024 Loss: 0.1966026094312468
2025 Loss: 0.15886659665923278
0.19965396419162887


In [3]:
### Incremental Learning

#Data

#read csv
training = pd.read_csv("train_data_m.csv")

#features
features = ['seed_dif', 
            'massey_rank_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]

#Initialize arrays
training_dfs = []
il_val_dfs = []
val_years = []

#Delete data before this year
first_year = 2003
training = training.query("Season >= @first_year").copy()

#Train base model up until this year
cutoff_year = 2019
years = [y for y in range(cutoff_year, 2025) if y != 2020]

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    seasons = [i]

    if i == cutoff_year:
        temp_training = training.query("Season <= @i")
    else:
        temp_training = training.query("Season in @seasons")
    
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    il_val_dfs.append(temp_val)
    val_years.append(val_year)



### Model

il_mod = XGBClassifier(n_estimators=400, max_depth=3, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, max_bin = 20, min_split_loss = 15, min_child_weight = 2, objective='binary:logistic', seed=323)

il_val_error = []

for i in range(len(years)):
    X_train = training_dfs[i][features]
    y_train = training_dfs[i]['result']

    val = il_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    if(i == 0):
        il_mod.fit(X_train, y_train)
    else:
        il_mod.set_params(n_estimators=il_mod.get_booster().num_boosted_rounds() + 50, max_depth = 3, learning_rate = 0.03)
        il_mod.fit(X_train, y_train, xgb_model=il_mod.get_booster()
    )

    #make predictions
    preds = il_mod.predict_proba(X_val)
    val['pred'] = preds[:,1]
    val['loss'] = (val['pred'] - val['result'])**2

    il_val_dfs[i] = val.copy()
    il_val_error.append(np.mean(val['loss']))
    print(val_years[i], "Loss:", np.mean(val['loss']), " |  Dif:", np.mean(val['loss']) - base_val_error[val_years[i] - 2017])


print("Last 5 Loss:", np.mean(il_val_error[-5:]), " | Dif:", np.mean(il_val_error[-5:]) - np.mean(base_val_error[-5:]))

#shap
explainer = shap.TreeExplainer(il_mod)
shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)

2021 Loss: 0.22149623720517952  |  Dif: 0.02589277198921547
2022 Loss: 0.22177646713961216  |  Dif: 0.03820388827336377
2023 Loss: 0.20187484610711617  |  Dif: -0.00036886601559882326
2024 Loss: 0.19872585583855276  |  Dif: 0.025252891248381343
2025 Loss: 0.1610636865229302  |  Dif: -0.0601212271630884
Last 5 Loss: 0.20098741856267816  | Dif: 0.0013334543710492863


In [4]:
#### Season as Feature

#read csv
training = pd.read_csv("train_data_m.csv")

#features
features = ['Season',
            'seed_dif', 
            'massey_rank_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            #'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            #'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]

#Initialize arrays
training_dfs = []
s_val_dfs = []
val_years = []
years = [2015,2016,2017,2018,2019,2021,2022,2023,2024, 2025]

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    temp_training = training.query("Season <= @i")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    s_val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

s_mod = XGBClassifier(n_estimators=600, max_depth=3, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 12, max_bin = 20, min_child_weight = 2, objective='binary:logistic', seed = 323)

s_val_error = []

for i in range(len(years)):
    X_train = training_dfs[i][features]
    y_train = training_dfs[i]['result']

    val = s_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    s_mod.fit(X_train, y_train)
    
    #make predictions
    preds = s_mod.predict_proba(X_val)
    val['pred'] = preds[:,1]
    val['loss'] = (val['pred'] - val['result'])**2

    s_val_dfs[i] = val.copy()
    s_val_error.append(np.mean(val['loss']))
    print(val_years[i], "Loss:", np.mean(val['loss']), " |  Dif:", np.mean(val['loss']) - base_val_error[val_years[i] - 2017])


print("Last 5 Loss:", np.mean(s_val_error[-5:]), " | Dif:", np.mean(s_val_error[-5:]) - np.mean(base_val_error[-5:]))

#shap
explainer = shap.TreeExplainer(s_mod)
#shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)


2016 Loss: 0.19532169132289753  |  Dif: 0.036455094663664755
2017 Loss: 0.18283335287613472  |  Dif: -0.014648048409927783
2018 Loss: 0.20229216086061685  |  Dif: 0.002249973803156502
2019 Loss: 0.1735392927531772  |  Dif: -0.030719462883657983
2021 Loss: 0.22126321872317367  |  Dif: 0.025659753507209615
2022 Loss: 0.22285151738036912  |  Dif: 0.03927893851412073
2023 Loss: 0.19972406399104317  |  Dif: -0.0025196481316718256
2024 Loss: 0.19652033603066893  |  Dif: 0.02304737144049751
2025 Loss: 0.15909453929299708  |  Dif: -0.062090374393021536
2026 Loss: nan  |  Dif: nan
Last 5 Loss: nan  | Dif: nan


c:\Users\will6\miniconda3\envs\cs320\Lib\site-packages\xgboost\core.py:750: UserWarning: [01:53:23] WARNING: D:\bld\xgboost-split_1768313916136\work\src\common\error_msg.cc:56: Empty dataset at worker: 0
  return func(**kwargs)


In [5]:
#Rolling

#read csv
training = pd.read_csv("train_data_m.csv")

#features
features = ['seed_dif', 
            'massey_rank_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            #'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            #'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]


#Initialize arrays
training_dfs = []
r_val_dfs = []
val_years = []
years = [2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2021,2022,2023,2024, 2025]

y_count = 8

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    if i <= 2019 or i >= 2020 + y_count:
        seasons = [i - k for k in range(y_count)]
    else:
        seasons = [i - k for k in range(y_count+1)]

    temp_training = training.query("Season in @seasons")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    r_val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

r_mod = XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 4, max_bin = 20, min_child_weight = 4, objective='binary:logistic', seed = 323)

r_val_error = []

for i in range(len(years)):
    X_train = training_dfs[i][features]
    y_train = training_dfs[i]['result']

    val = r_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    r_mod.fit(X_train, y_train)
    
    #make predictions
    preds = r_mod.predict_proba(X_val)
    val['pred'] = preds[:,1]
    val['loss'] = (val['pred'] - val['result'])**2

    r_val_dfs[i] = val.copy()
    r_val_error.append(np.mean(val['loss']))
    print(val_years[i], "Loss:", np.mean(val['loss']), " |  Dif:", np.mean(val['loss']) - base_val_error[val_years[i] - 2017])

print("Last 5 Loss:", np.mean(r_val_error[-5:]), " | Dif:", np.mean(r_val_error[-5:]) - np.mean(base_val_error[-5:]))

#shap
explainer = shap.TreeExplainer(r_mod)
#shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)


2011 Loss: 0.22553568490739476  |  Dif: 0.05206272031722334
2012 Loss: 0.19950623903519016  |  Dif: -0.021678674650828456
2013 Loss: 0.19490476597835282  |  Dif: -0.02773487874467581
2014 Loss: 0.20900683072203602  |  Dif: 0.010030774263418446
2015 Loss: 0.1722514512334501  |  Dif: -0.024351158197796707
2016 Loss: 0.20399947568629492  |  Dif: 0.045132879027062145
2017 Loss: 0.1807728205447766  |  Dif: -0.016708580741285894
2018 Loss: 0.2085208569208964  |  Dif: 0.008478669863436039
2019 Loss: 0.17343607541471237  |  Dif: -0.03082268022212281
2021 Loss: 0.22552935936159405  |  Dif: 0.02992589414562999
2022 Loss: 0.22587202132017614  |  Dif: 0.04229944245392775
2023 Loss: 0.19449260453120384  |  Dif: -0.007751107591511153
2024 Loss: 0.19627577230652507  |  Dif: 0.02280280771635365
2025 Loss: 0.15518431750804532  |  Dif: -0.0660005961779733
2026 Loss: nan  |  Dif: nan
Last 5 Loss: nan  | Dif: nan


In [6]:
#### Weights

#read csv
training = pd.read_csv("train_data_m.csv")

#features
features = ['seed_dif', 
            'massey_rank_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]

#Initialize arrays
training_dfs = []
w_val_dfs = []
val_years = []
years = [2011,2012,2013,2014,2015,2016,2017,2018,2019,2021,2022,2023,2024]

#Weight parameter
weight_param = 0.95

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    temp_training = training.query("Season <= @i")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    w_val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

w_mod = XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 6, max_bin = 20, min_child_weight = 2, objective='binary:logistic', seed = 323)

w_val_error = []

for i in range(len(years)):
    temp_df = training_dfs[i]
    temp_df = temp_df.assign(weight = (weight_param ** (temp_df['Season'].max() - temp_df['Season'])).clip(0.1))

    X_train = temp_df[features]
    y_train = temp_df['result']
    train_weights = temp_df['weight']

    val = w_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    w_mod.fit(X_train, y_train, sample_weight=train_weights)
    
    #make predictions
    preds = w_mod.predict_proba(X_val)
    val['pred'] = preds[:,1]
    val['loss'] = (val['pred'] - val['result'])**2

    w_val_dfs[i] = val.copy()
    w_val_error.append(np.mean(val['loss']))
    print(val_years[i], "Loss:", np.mean(val['loss']), " |  Dif:", np.mean(val['loss']) - base_val_error[val_years[i] - 2017])

print("Last 5 Loss:", np.mean(w_val_error[-5:]), " | Dif:", np.mean(w_val_error[-5:]) - np.mean(base_val_error[-5:]))

#shap
explainer = shap.TreeExplainer(w_mod)
#shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)


2012 Loss: 0.1993957849423377  |  Dif: -0.021789128743680913
2013 Loss: 0.1976038274781247  |  Dif: -0.025035817244903946
2014 Loss: 0.20519535744443526  |  Dif: 0.006219300985817688
2015 Loss: 0.17459440854376498  |  Dif: -0.022008200887481816
2016 Loss: 0.19874537580837823  |  Dif: 0.03987877914914545
2017 Loss: 0.18160378505596517  |  Dif: -0.01587761623009734
2018 Loss: 0.20339080020509484  |  Dif: 0.0033486131476344883
2019 Loss: 0.1712323797296404  |  Dif: -0.03302637590719479
2021 Loss: 0.21826736850664274  |  Dif: 0.02266390329067869
2022 Loss: 0.22170751936205568  |  Dif: 0.03813494049580729
2023 Loss: 0.19832019212640753  |  Dif: -0.003923519996307467
2024 Loss: 0.19776060706669785  |  Dif: 0.024287642476526428
2025 Loss: 0.15931221442944518  |  Dif: -0.06187269925657343
Last 5 Loss: 0.19907358029824979  | Dif: -0.0005803838933790828


In [7]:
### Combine

#Merge Data
def get_vals(lst, prefix):
    df = pd.concat(lst, ignore_index=True)[['Season', 'DayNum', 'team_A', 'team_B', 'score_A', 'score_B', 'result', 'pred', 'loss']]
    df = df.rename({'pred': f'{prefix}pred', 'loss': f'{prefix}loss'}, axis='columns')
    return df

base_val = get_vals(base_val_dfs, "base_")
il_val = get_vals(il_val_dfs, "il_")
s_val = get_vals(s_val_dfs, "s_")
r_val = get_vals(r_val_dfs, "r_")
w_val = get_vals(w_val_dfs, "w_")

merge_keys = ["Season", 'DayNum', 'team_A', 'team_B', 'score_A', 'score_B', 'result']

full_val = pd.merge(base_val, il_val, how="outer", on=merge_keys).merge(
        s_val, how="outer", on=merge_keys).merge(
        r_val, how="outer", on=merge_keys).merge(
        w_val, how="outer", on=merge_keys)


full_val.tail()


#combined prediction
base_weight = 0.4
il_weight = 0
s_weight = 0
r_weight = 0.3
w_weight = 0.3

full_val = full_val.assign(com_pred = full_val['base_pred']*base_weight + full_val['il_pred']*il_weight + full_val['s_pred']*s_weight + full_val['r_pred']*r_weight + full_val['w_pred']*w_weight)
full_val['com_loss'] = (full_val['com_pred'] - full_val['result'])**2

full_val.tail()

#group and summarize
val_summary = full_val.groupby(["Season"]).agg(
    base_val=("base_loss", "mean"),
    il_val=("il_loss", "mean"),
    s_val=("s_loss", "mean"),
    r_val=("r_loss", "mean"),
    w_val=("w_loss", "mean"),
    com_val=("com_loss", "mean")).reset_index()

val_summary.loc['last5'] = val_summary.set_index('Season').iloc[-5:].mean()

val_summary.tail(6)

,Season,base_val,il_val,s_val,r_val,w_val,com_val
9,2021.0,0.221185,0.221496,0.221263,0.225529,0.218267,0.220995
10,2022.0,0.222640,0.221776,0.222852,0.225872,0.221708,0.222543
11,2023.0,0.198976,0.201875,0.199724,0.194493,0.198320,0.196542
12,2024.0,0.196603,0.198726,0.196520,0.196276,0.197761,0.196395
13,2025.0,0.158867,0.161064,0.159095,0.155184,0.159312,0.157405
last5,NaN,0.199654,0.200987,0.199891,0.199471,0.199074,0.198776


In [8]:
data = pd.read_csv("data_for_submission_m.csv")

#read csv
training = pd.read_csv("train_data_m.csv")

#features
base_features = ['seed_dif', 
            'massey_rank_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            #'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            #'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]

r_features = ['seed_dif', 
            'massey_rank_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            #'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            #'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]

#features
w_features = ['seed_dif', 
            'massey_rank_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]


base_training = training.query("Season <= 2025")
r_training = training.query("Season <= 2025").query("Season >= 2017")
w_training = training.query("Season <= 2025")

#model
base_mod = XGBClassifier(n_estimators=600, max_depth=3, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 12, max_bin = 20, min_child_weight = 2, num_parallel_tree = 1, objective='binary:logistic', seed = 323)
r_mod = XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 4, max_bin = 20, min_child_weight = 4, objective='binary:logistic', seed = 323)
w_mod = XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 6, max_bin = 20, min_child_weight = 2, objective='binary:logistic', seed = 323)

#training data
X_train_base = base_training[base_features]
y_train_base = base_training['result']

X_train_r = r_training[r_features]
y_train_r = r_training['result']

w_training = w_training.assign(weight = (0.95 ** (w_training['Season'].max() - w_training['Season'])).clip(0.1))
X_train_w = w_training[w_features]
y_train_w = w_training['result']
train_weights = w_training['weight']

#fit model
base_mod.fit(X_train_base, y_train_base)
r_mod.fit(X_train_r, y_train_r)
w_mod.fit(X_train_w, y_train_w, sample_weight=train_weights)
    
#data for making preds
X_data_base = data[base_features]
X_data_r = data[r_features]
X_data_w = data[w_features]

base_preds = base_mod.predict_proba(X_data_base)
r_preds = r_mod.predict_proba(X_data_r)
w_preds = w_mod.predict_proba(X_data_w)

#make predictions
base_preds = base_mod.predict_proba(X_data_base)
r_preds = r_mod.predict_proba(X_data_r)
w_preds = w_mod.predict_proba(X_data_w)

#put preds in df
data['base_pred'] = base_preds[:,1]
data['r_pred'] = r_preds[:,1]
data['w_pred'] = w_preds[:,1]

data = data.assign(com_pred = data['base_pred']*0.4 + data['r_pred']*0.3 + data['w_pred']*0.3)


In [9]:
### Submission Files

com_submission_file = data.assign(
        ID="2026_" + data['team_A'].astype(str) + "_" + data['team_B'].astype(str).copy()
    ).assign(Pred = data['com_pred'])[['ID', 'Pred']]

base_submission_file = data.assign(
        ID="2026_" + data['team_A'].astype(str) + "_" + data['team_B'].astype(str).copy()
    ).assign(Pred = data['base_pred'])[['ID', 'Pred']]

r_submission_file = data.assign(
        ID="2026_" + data['team_A'].astype(str) + "_" + data['team_B'].astype(str).copy()
    ).assign(Pred = data['r_pred'])[['ID', 'Pred']]

w_submission_file = data.assign(
        ID="2026_" + data['team_A'].astype(str) + "_" + data['team_B'].astype(str).copy()
    ).assign(Pred = data['w_pred'])[['ID', 'Pred']]



com_submission_file.to_csv("submission_files/men_com_submission_file.csv", index=False)
base_submission_file.to_csv("submission_files/men_base_submission_file.csv", index=False)
r_submission_file.to_csv("submission_files/men_r_submission_file.csv", index=False)
w_submission_file.to_csv("submission_files/men_w_submission_file.csv", index=False)

In [10]:
### Main Submission
main_submission = data.assign(
        ID="2026_" + data['team_A'].astype(str) + "_" + data['team_B'].astype(str).copy()
    ).assign(Pred = data['com_pred'])

main_submission.loc[main_submission['seed_dif'] >= 9, 'Pred'] = 0
main_submission.loc[main_submission['seed_dif'] <= -9, 'Pred'] = 1

main_submission = main_submission[['ID', 'Pred']]

main_submission.to_csv("submission_files/men_main_submission_file.csv", index=False)
